# จำลองกะการผลิตในโรงงาน (1 กะ = 40 ถาด)

**สถานการณ์:** ไลน์บรรจุอะไหล่ ทุกถาดต้องมี**ชิ้นแดงใหญ่ 5 ชิ้น** กล้อง Basler ติดเหนือสายพาน ถ่ายทุกถาดที่วิ่งผ่าน (Trigger)

สิ่งที่เกิดขึ้นจริงระหว่างกะ และถูกจำลองไว้ใน notebook นี้:

| เหตุการณ์ | ในโรงงานจริง |
|---|---|
| **แสงเปลี่ยนตามเวลา** | ถาดที่ 1–15 เช้า แสงปกติ  ถาดที่ 16–28 บ่าย เมฆบังหน้าต่าง  ถาดที่ 29–40 เย็น ไฟโรงงานดวงหนึ่งดับ |
| **ชิ้นงานวิ่ง** | สายพานเคลื่อนตลอด ภาพเบลอถ้า Exposure นาน |
| **ของเสีย 4 แบบ** | ชิ้นขาด, ชิ้นเกิน, เศษชิ้นแตกปน, สิ่งแปลกปลอมสีส้มปน |

เรารู้ "เฉลย" ของทุกถาดจากไฟล์ `factory_labels.csv` จึงวัดได้ว่าระบบตรวจ**ผิด**กี่ครั้ง:
- **คัดทิ้งผิด (False Reject):** ถาดดีแต่ระบบบอก NG → ทิ้งของดี เสียเงิน
- **หลุดรอด (Escape):** ถาดเสียแต่ระบบบอก OK → ส่งถึงลูกค้า **ร้ายแรงที่สุด**

In [ ]:
import sys, os, csv, time
sys.path.insert(0, os.path.abspath("../scripts"))
import cv2
import numpy as np
import matplotlib.pyplot as plt
from vision_camera import show, EmuLine, simulate_real, img_path

TRUTH = {int(r["part"]): (r["truth"], r["defect"])
         for r in csv.DictReader(open(img_path("factory_labels.csv"), encoding="utf-8"))}
print("โหลดเฉลย", len(TRUTH), "ถาด  ของดี", sum(t == "OK" for t, _ in TRUTH.values()), "ถาด")

## 1. ฟังก์ชันของไลน์ผลิต
อ่านคร่าวๆ ได้ ไม่ต้องแก้ cell นี้

In [ ]:
def light_at(i):
    """ความสว่างของแสงรอบไลน์ ตามช่วงเวลาของกะ"""
    return 1.0 if i <= 15 else (0.5 if i <= 28 else 0.2)

def inspect(img, s):
    """ตรวจถาด 1 ใบ ตามค่าตั้ง s  คืน (ผล, จำนวนชิ้นใหญ่, จำนวนเศษ, จำนวนสิ่งแปลกปลอม)"""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    red = cv2.inRange(hsv, np.array([0, s["s_min"], s["v_min"]]), np.array([10, 255, 255]))
    cs, _ = cv2.findContours(red, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    areas = [cv2.contourArea(c) for c in cs]
    big   = sum(a > s["min_area"] for a in areas)
    chips = sum(50 < a <= s["min_area"] for a in areas)
    orange = cv2.inRange(hsv, np.array([12, s["s_min"], s["v_min"]]), np.array([22, 255, 255]))
    co, _ = cv2.findContours(orange, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    foreign = sum(cv2.contourArea(c) > s["min_area"] for c in co)

    ng = big != 5                                     # กฎหลัก: ต้องมี 5 ชิ้น
    if s["check_chips"]:   ng = ng or chips > 0       # กฎเสริม: ห้ามมีเศษชิ้นแตก
    if s["check_foreign"]: ng = ng or foreign > 0     # กฎเสริม: ห้ามมีสิ่งแปลกปลอม
    return ("NG" if ng else "OK"), big, chips, foreign

def run_shift(s, show_mistakes=True):
    """เดินไลน์ 1 กะ 40 ถาด แล้วสรุปผล"""
    line = EmuLine("factory")
    rows = []
    t0 = time.perf_counter()
    try:
        for i in range(1, 41):
            frame = line.trigger()                                   # ถาดมาถึง -> Trigger
            L = 1.0 if s["enclosure"] else light_at(i)
            exp = s["exposure_ms"]
            img = simulate_real(frame, exp, s["gain_db"], L, s["speed"], seed=i)
            if s["auto_exposure"]:                                   # ปรับ Exposure ตามความสว่างจริง
                exp = min(20.0, exp * 235.0 / max(float(np.median(img)), 1.0))
                img = simulate_real(frame, exp, s["gain_db"], L, s["speed"], seed=i)
            verdict, big, chips, foreign = inspect(img, s)
            truth, defect = TRUTH[i]
            rows.append(dict(i=i, truth=truth, defect=defect, verdict=verdict,
                             big=big, light=L, exp=exp, img=img))
    finally:
        line.close()
    sec = time.perf_counter() - t0

    fr  = [r for r in rows if r["truth"] == "OK" and r["verdict"] == "NG"]
    esc = [r for r in rows if r["truth"] == "NG" and r["verdict"] == "OK"]
    good = 40 - len(fr) - len(esc)
    cost = len(fr) * s["cost_false_reject"] + len(esc) * s["cost_escape"]

    print(f"ตรวจถูก {good}/40 ถาด   คัดทิ้งผิด {len(fr)}   หลุดรอด {len(esc)}")
    print(f"ความเสียหายโดยประมาณ {cost:,} บาท   (ทิ้งของดี {s['cost_false_reject']} บาท/ถาด, ลูกค้าเคลม {s['cost_escape']:,} บาท/ถาด)")
    print(f"เวลาตรวจเฉลี่ย {sec / 40 * 1000:.0f} ms/ถาด")

    # กราฟ: แสงตลอดกะ และจุดที่ตรวจผิด
    fig, ax = plt.subplots(figsize=(12, 2.8))
    ax.plot([r["i"] for r in rows], [r["light"] for r in rows], color="gray", label="แสงรอบไลน์")
    for r in fr:  ax.scatter(r["i"], r["light"], color="tab:orange", s=80, zorder=3)
    for r in esc: ax.scatter(r["i"], r["light"], color="tab:red", marker="x", s=100, zorder=3)
    ax.set_xlabel("ถาดที่"); ax.set_ylim(0, 1.2)
    ax.set_title("จุดส้ม = คัดทิ้งผิด   กากบาทแดง = หลุดรอด")
    plt.show()

    if show_mistakes and (fr or esc):
        bad = (esc + fr)[:3]
        show(*[r["img"] for r in bad],
             titles=[f"ถาด {r['i']}: จริง {r['truth']} ({r['defect']}) ระบบบอก {r['verdict']}" for r in bad], size=4)
    return rows

## 2. ค่าตั้งของระบบตรวจ
ทุกภารกิจจะแก้ค่าใน cell นี้ แล้วรัน cell ข้อ 3 ใหม่

In [ ]:
SETTINGS = dict(
    # --- กล้อง ---
    exposure_ms   = 5.0,     # เวลาเปิดรับแสง
    gain_db       = 0.0,     # ขยายสัญญาณ
    auto_exposure = False,   # ให้ปรับ Exposure เองตามความสว่าง
    # --- หน้างาน ---
    enclosure     = False,   # ครอบกล่องกันแสงภายนอก + ใช้ไฟของตัวเอง
    speed         = 1.0,     # ความเร็วสายพาน (พิกเซล/มิลลิวินาที)
    # --- โปรแกรมตรวจ ---
    s_min = 50, v_min = 50, min_area = 1000,
    check_chips   = False,   # กฎเสริม: ตรวจเศษชิ้นแตก
    check_foreign = False,   # กฎเสริม: ตรวจสิ่งแปลกปลอมสีส้ม
    # --- ต้นทุน ---
    cost_false_reject = 20,      # บาท ต่อถาดดีที่ทิ้งไป
    cost_escape       = 2000,    # บาท ต่อถาดเสียที่หลุดถึงลูกค้า
)

## 3. เดินไลน์ 1 กะ

In [ ]:
rows = run_shift(SETTINGS)

## ภารกิจ

**ภารกิจ 1: ดูปัญหา** รันค่าเริ่มต้น ระบบผิดกี่ถาด? ผิดช่วงเวลาไหน? แบบไหน?
> เฉลย: ตรวจถูก 28/40  คัดทิ้งผิด 8 (ช่วงเย็นแสงน้อย)  หลุดรอด 4 (เศษชิ้นแตก + สิ่งแปลกปลอม)  เสียหาย 8,160 บาท

**ภารกิจ 2: แก้ด้วยกล้องอย่างเดียว** ตั้ง `gain_db = 12` หรือ `exposure_ms = 15` แล้วรันใหม่
> ตรวจถูกเหลือ 17/40  ช่วงเย็นดีขึ้น แต่ช่วงเช้าและบ่าย**พังแทน** เพราะสว่างเกิน  **ค่ากล้องค่าเดียวใช้ทั้งกะไม่ได้ ถ้าแสงรอบไลน์เปลี่ยน**

**ภารกิจ 3: แก้ที่ต้นเหตุ คุมแสง** คืนค่ากล้องเดิม แล้วตั้ง `enclosure = True`
> ตรวจถูก 34/40 คัดทิ้งผิดเหลือ 0  นี่คือเหตุผลที่สถานีตรวจในโรงงานมักมีกล่องครอบและไฟของตัวเอง  แต่ยังหลุดรอด 6 ถาด เพราะยังไม่ได้ตรวจเศษกับสิ่งแปลกปลอม
> **สังเกต:** ตรวจถูกมากขึ้น แต่ความเสียหายกลับ**เพิ่ม**เป็น 12,000 บาท เพราะของหลุดถึงลูกค้าแพงกว่าทิ้งของดี 100 เท่า  ดูแค่ "% ตรวจถูก" จึงไม่พอ

**ภารกิจ 4: เพิ่มกฎตรวจ** ตั้ง `check_chips = True` และ `check_foreign = True`
> ตรวจถูก 40/40 เสียหาย 0 บาท  **ระบบตรวจที่ดี = แสงคงที่ + กฎตรวจครบทุกแบบของเสียที่รู้จัก**

**ภารกิจ 5 (ท้าทาย): ถ้าครอบกล่องไม่ได้** ตั้ง `enclosure = False` และ `auto_exposure = True` (เปิดกฎเสริมไว้)
> ระบบปรับ Exposure เองทุกถาด ก็ตรวจถูกครบ  แต่ลองตั้ง `speed = 2` (สายพานเร็วขึ้น) แล้วดูว่าช่วงเย็นเกิดอะไรขึ้น และเพราะอะไร
> เฉลย: ผิด 1 ถาดช่วงเย็น เพราะแสงน้อย ระบบต้องเปิด Exposure นาน ชิ้นงานที่วิ่งเร็วจึงเบลอ  ไลน์เร็วจึงต้องการไฟแรง ไม่ใช่ Exposure นาน

**คำถามสำหรับผู้บริหาร:** ลองเปลี่ยน `cost_escape` ให้ตรงกับความเสียหายจริงของโรงงานท่าน ระบบตรวจแบบไหนคุ้มค่าที่สุด?

---
# ลงมือทำ: วิศวกรออกแบบสถานีตรวจ (3 ภารกิจ)
ข้างบนลองทีละค่า  ตอนนี้ใช้โค้ด**ค้นหาค่าที่ดีที่สุด**แบบที่วิศวกรทำจริง  เฉลยอยู่ท้าย notebook

In [ ]:
def check(title, got, expected):
    ok = got == expected
    print(('✅ ผ่าน  ' if ok else '❌ ยัง   ') + title + f'   ได้ {got}  ต้องการ {expected}')
    return ok

def evaluate(s):
    # เดินไลน์ 1 กะแบบเงียบ (ไม่วาดกราฟ) คืน (คัดทิ้งผิด, หลุดรอด)
    line = EmuLine('factory'); fr = esc = 0
    try:
        for i in range(1, 41):
            frame = line.trigger()
            L = 1.0 if s['enclosure'] else light_at(i)
            exp = s['exposure_ms']
            img = simulate_real(frame, exp, s['gain_db'], L, s['speed'], seed=i)
            if s['auto_exposure']:
                exp = min(20.0, exp * 235.0 / max(float(np.median(img)), 1.0))
                img = simulate_real(frame, exp, s['gain_db'], L, s['speed'], seed=i)
            verdict = inspect(img, s)[0]
            truth = TRUTH[i][0]
            fr += truth == 'OK' and verdict == 'NG'
            esc += truth == 'NG' and verdict == 'OK'
    finally:
        line.close()
    return fr, esc

BASE = dict(SETTINGS)
BASE.update(exposure_ms=5.0, gain_db=0.0, auto_exposure=False, enclosure=False, speed=1.0,
            s_min=50, v_min=50, min_area=1000, check_chips=False, check_foreign=False)
print('ค่าเริ่มต้น (คัดทิ้งผิด, หลุดรอด):', evaluate(BASE))

## ภารกิจ 1: ตรวจถูก 40/40 โดย**ไม่ครอบกล่อง**
ตั้งค่า `MY` (ห้ามใช้ `enclosure=True`) ให้คัดทิ้งผิด 0 และหลุดรอด 0

In [ ]:
MY = dict(BASE)
MY.update()                                        # TODO: ใส่ค่าที่ต้องการ เช่น MY.update(auto_exposure=True, ...)
result = evaluate(MY)
check('ภารกิจ 1 ไม่ครอบกล่อง', MY['enclosure'], False)
check('ภารกิจ 1 (คัดทิ้งผิด, หลุดรอด)', result, (0, 0))

## ภารกิจ 2: เลือกแบบที่คุ้มที่สุด
มี 3 แบบให้เลือก  เขียน `cost()` คิดความเสียหาย: คัดทิ้งผิดถาดละ 20 บาท  หลุดรอดถาดละ 2,000 บาท  แล้วหาแบบที่ถูกที่สุด
(ในงานจริงต้องบวกค่าอุปกรณ์ด้วย เช่น กล่องครอบ + ไฟ ราว 15,000 บาท ครั้งเดียว)

In [ ]:
OPTIONS = {
    'A ค่าเริ่มต้น':          dict(BASE),
    'B เพิ่ม Gain 12':        dict(BASE, gain_db=12.0),
    'C ครอบกล่อง + กฎครบ':    dict(BASE, enclosure=True, check_chips=True, check_foreign=True),
}
def cost(fr, esc):
    return 0                                       # TODO

table = {name: cost(*evaluate(s)) for name, s in OPTIONS.items()}
for name, c in table.items():
    print(f'{name:24s} {c:>8,} บาท / กะ')
best = None                                        # TODO: ชื่อแบบที่ cost ต่ำที่สุด (ใช้ min กับ key)
check('ภารกิจ 2 ความเสียหายแบบ A', table['A ค่าเริ่มต้น'], 8160)
check('ภารกิจ 2 แบบที่คุ้มที่สุด', best, 'C ครอบกล่อง + กฎครบ')

## ภารกิจ 3: สายพานเร่งได้เร็วสุดเท่าไร
ใช้แบบ **ไม่ครอบกล่อง + Auto Exposure + กฎครบ**  ลองความเร็ว 1.0, 1.5, 2.0, 2.5, 3.0  หาความเร็วสูงสุดที่ยังตรวจถูกครบ 40/40
นี่คือคำถามที่ฝ่ายผลิตถามเสมอ: “ถ้าเพิ่มกำลังผลิต ระบบตรวจยังไหวไหม”

In [ ]:
FAST = dict(BASE, auto_exposure=True, check_chips=True, check_foreign=True)
max_speed = None
for sp in [1.0, 1.5, 2.0, 2.5, 3.0]:
    fr, esc = evaluate(dict(FAST, speed=sp))
    print(f'ความเร็ว {sp}:  คัดทิ้งผิด {fr}  หลุดรอด {esc}')
    # TODO: ถ้าไม่มีผิดเลย ให้จำความเร็วนี้ไว้ใน max_speed
check('ภารกิจ 3 ความเร็วสูงสุดที่ยังถูกครบ', max_speed, 1.5)
print('ถ้าต้องการเร็วกว่านี้ ต้องเพิ่มแสง (ครอบกล่อง + ไฟแรง) เพื่อใช้ Exposure สั้นลง ไม่ใช่เพิ่ม Exposure')

<details>
<summary><b>เฉลย (คลิกเพื่อเปิด)</b></summary>

**ภารกิจ 1:** `MY.update(auto_exposure=True, check_chips=True, check_foreign=True)`

**ภารกิจ 2:**
```python
def cost(fr, esc):
    return fr * 20 + esc * 2000
...
best = min(table, key=table.get)
```

**ภารกิจ 3:**
```python
    if fr == 0 and esc == 0:
        max_speed = sp
```
</details>